In [1]:
from pathlib import Path
import math

import numpy as np
import pandas as pd
from prophet import Prophet  # make sure 'prophet' is installed
from sklearn.metrics import mean_squared_error, mean_absolute_error

INPUT_CSV = Path("data_clean/forecast/sltda_arrivals_monthly.csv")
OUTPUT_CSV = Path("data_clean/forecast/sltda_arrivals_prophet_forecast.csv")
MODELS_DIR = Path("models")
MODELS_DIR.mkdir(parents=True, exist_ok=True)


e:\Research_Final\component3_service\prophet_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# ARIMA Arrivals Forecasting - Notebook Version
import os
os.chdir(r"E:\Research_Final\component3_service")
print("New CWD:", os.getcwd())

import math
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from pandas.tseries.offsets import MonthBegin
from statsmodels.tsa.arima.model import ARIMA

# Paths
INPUT_CSV = Path("data_clean/forecast/sltda_arrivals_monthly.csv")
OUTPUT_CSV = Path("data_clean/forecast/sltda_arrivals_arima_forecast.csv")


New CWD: E:\Research_Final\component3_service


In [4]:
def load_arrivals_for_prophet(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path, parse_dates=["date"])
    df = df.sort_values("date")

    # Handle COVID collapse same as ARIMA
    df["total_arrivals"] = df["total_arrivals"].astype(float)
    df.loc[df["total_arrivals"] < 1000, "total_arrivals"] = np.nan
    df["total_arrivals"] = df["total_arrivals"].interpolate(method="linear")

    print(
        f"Loaded {len(df)} monthly points from "
        f"{df['date'].min().date()} to {df['date'].max().date()}"
    )
    return df

df_arrivals = load_arrivals_for_prophet(INPUT_CSV)
df_arrivals.head()


Loaded 94 monthly points from 2018-01-01 to 2025-10-01


,date,year,month,month_name,total_arrivals
0,2018-01-01,2018,1,Jan,152377.0
1,2018-01-02,2018,2,Feb,158782.0
2,2018-01-03,2018,3,Mar,160784.0
3,2018-01-04,2018,4,Apr,128013.0
4,2018-01-05,2018,5,May,95794.0


In [5]:
VAL_MONTHS = 12

# Prophet expects columns 'ds' (date) and 'y' (target)
df_prophet = df_arrivals[["date", "total_arrivals"]].rename(
    columns={"date": "ds", "total_arrivals": "y"}
)

# Last 12 months as validation
train_df = df_prophet.iloc[:-VAL_MONTHS].copy()
val_df   = df_prophet.iloc[-VAL_MONTHS:].copy()

print("Train period:", train_df["ds"].min().date(), "→", train_df["ds"].max().date())
print("Val period  :", val_df["ds"].min().date(),   "→", val_df["ds"].max().date())


Train period: 2018-01-01 → 2024-10-01
Val period  : 2024-11-01 → 2025-10-01


In [6]:
def fit_prophet_model(train_df: pd.DataFrame) -> Prophet:
    m = Prophet(
        yearly_seasonality=True,
        weekly_seasonality=False,
        daily_seasonality=False,
        seasonality_mode="additive",   # you can try "multiplicative" later
    )
    m.fit(train_df)
    return m

def evaluate_prophet(train_df: pd.DataFrame, val_df: pd.DataFrame):
    m = fit_prophet_model(train_df)

    # Create a dataframe of all dates up to the end of validation
    last_val_date = val_df["ds"].max()
    future = m.make_future_dataframe(periods=0, freq="MS")  # only train dates
    # For predictions, concat train+val horizon:
    future = pd.concat([train_df[["ds"]], val_df[["ds"]]]).drop_duplicates().sort_values("ds")

    forecast = m.predict(future)

    # Extract predictions only for validation window
    forecast_val = forecast[forecast["ds"].isin(val_df["ds"])]
    y_true = val_df["y"].values
    y_pred = forecast_val["yhat"].values

    rmse = math.sqrt(mean_squared_error(y_true, y_pred))
    mae  = mean_absolute_error(y_true, y_pred)

    print(
        f"[Prophet] Validation window: {val_df['ds'].min().date()} → {val_df['ds'].max().date()}"
    )
    print(f"[Prophet] Validation RMSE: {rmse:.2f}")
    print(f"[Prophet] Validation MAE : {mae:.2f}")

    return m, forecast, rmse, mae

prophet_model, prophet_forecast_full, prophet_rmse, prophet_mae = evaluate_prophet(
    train_df, val_df
)


14:39:00 - cmdstanpy - INFO - Chain [1] start processing
14:39:00 - cmdstanpy - INFO - Chain [1] done processing
14:39:00 - cmdstanpy - ERROR - Chain [1] error: code '3221225785' 


RuntimeError: Error during optimization! Command 'E:\Research_Final\component3_service\prophet_env\Lib\site-packages\prophet\stan_model\prophet_model.bin random seed=86711 data file=C:\Users\ASUS\AppData\Local\Temp\tmpqhatlkvm\1fihqgyu.json init=C:\Users\ASUS\AppData\Local\Temp\tmpqhatlkvm\gj2re46e.json output file=C:\Users\ASUS\AppData\Local\Temp\tmpqhatlkvm\prophet_modelm44dwxzc\prophet_model-20251123143900.csv method=optimize algorithm=newton iter=10000' failed: 

In [7]:
from prophet import Prophet
import pandas as pd
import numpy as np
import math
from sklearn.metrics import mean_squared_error, mean_absolute_error

def fit_prophet_model(train_df: pd.DataFrame) -> Prophet:
    # Work on a copy
    df = train_df.copy()

    # Keep only required columns
    df = df[["ds", "y"]]

    # Ensure ds is datetime
    df["ds"] = pd.to_datetime(df["ds"])

    # Ensure y is numeric
    df["y"] = pd.to_numeric(df["y"], errors="coerce")

    # Drop NaNs / inf
    df = df.replace([np.inf, -np.inf], np.nan).dropna(subset=["ds", "y"])

    # Sort by date & drop duplicate dates
    df = df.sort_values("ds")
    df = df[~df["ds"].duplicated()]

    print(f"[Prophet] Training on {len(df)} rows from {df['ds'].min().date()} to {df['ds'].max().date()}")

    m = Prophet(
        yearly_seasonality=True,
        weekly_seasonality=False,
        daily_seasonality=False,
        seasonality_mode="additive",
    )

    # If optimization is touchy, you *can* force a different algorithm:
    # m.fit(df, algorithm="LBFGS")
    m.fit(df)
    return m


In [8]:
def evaluate_prophet(train_df: pd.DataFrame, val_df: pd.DataFrame):
    # Fit on cleaned train_df
    m = fit_prophet_model(train_df)

    # Build future dates as union of train + val ds
    future = pd.concat(
        [train_df[["ds"]], val_df[["ds"]]],
        axis=0
    ).drop_duplicates().sort_values("ds")

    forecast = m.predict(future)

    # Extract predictions only for validation window
    forecast_val = forecast[forecast["ds"].isin(val_df["ds"])].sort_values("ds")
    val_df_sorted = val_df.sort_values("ds")

    y_true = val_df_sorted["y"].values
    y_pred = forecast_val["yhat"].values

    rmse = math.sqrt(mean_squared_error(y_true, y_pred))
    mae  = mean_absolute_error(y_true, y_pred)

    print(
        f"[Prophet] Validation window: {val_df_sorted['ds'].min().date()} → {val_df_sorted['ds'].max().date()}"
    )
    print(f"[Prophet] Validation RMSE: {rmse:.2f}")
    print(f"[Prophet] Validation MAE : {mae:.2f}")

    return m, forecast, rmse, mae


In [9]:
prophet_model, prophet_forecast_full, prophet_rmse, prophet_mae = evaluate_prophet(
    train_df, val_df
)


[Prophet] Training on 82 rows from 2018-01-01 to 2024-10-01


14:39:25 - cmdstanpy - INFO - Chain [1] start processing
14:39:25 - cmdstanpy - INFO - Chain [1] done processing
14:39:25 - cmdstanpy - ERROR - Chain [1] error: code '3221225785' 


RuntimeError: Error during optimization! Command 'E:\Research_Final\component3_service\prophet_env\Lib\site-packages\prophet\stan_model\prophet_model.bin random seed=12812 data file=C:\Users\ASUS\AppData\Local\Temp\tmpqhatlkvm\uplyy38p.json init=C:\Users\ASUS\AppData\Local\Temp\tmpqhatlkvm\vyqtog6d.json output file=C:\Users\ASUS\AppData\Local\Temp\tmpqhatlkvm\prophet_model4w_cey1q\prophet_model-20251123143925.csv method=optimize algorithm=newton iter=10000' failed: 

In [10]:
from prophet import Prophet
import pandas as pd
import numpy as np
import math
from sklearn.metrics import mean_squared_error, mean_absolute_error

def fit_prophet_model(train_df: pd.DataFrame) -> Prophet:
    """
    Fit a Prophet model on train_df (expects columns: ds, y).
    Handles common optimization failures by retrying with relaxed settings.
    """
    # Clean + ensure correct types
    df = train_df.copy()[["ds", "y"]]
    df["ds"] = pd.to_datetime(df["ds"])
    df = df.sort_values("ds")
    df["y"] = df["y"].astype(float)
    df = df.dropna()

    # Base model config (monthly-style data: no weekly/daily)
    base_kwargs = dict(
        yearly_seasonality=True,
        weekly_seasonality=False,
        daily_seasonality=False,
        seasonality_mode="additive",
    )

    m = Prophet(**base_kwargs)

    try:
        # Default fit (uses L-BFGS with Newton fallback internally)
        print("[Prophet] Fitting model with default optimizer...")
        m.fit(df)
    except RuntimeError as e:
        print("\n[Prophet] ⚠ Optimization failed with default settings.")
        print("[Prophet] Error was:\n", e)
        print("[Prophet] Retrying with relaxed convergence (require_converged=False)...")

        # Recreate model (clean state) and relax convergence
        m = Prophet(**base_kwargs)
        m.fit(
            df,
            algorithm="LBFGS",
            iter=2000,
            require_converged=False,   # <-- don't raise error if not fully converged
        )

    return m


def evaluate_prophet(train_df: pd.DataFrame, val_df: pd.DataFrame):
    """
    Fit Prophet on train_df, forecast over train+val window,
    and compute RMSE/MAE on the validation period.
    train_df, val_df must have columns: ds (datetime-like), y (numeric).
    """
    # 1. Fit model (robust, with retry)
    m = fit_prophet_model(train_df)

    # 2. Build future dates: union of train + val ds
    future = (
        pd.concat([train_df[["ds"]], val_df[["ds"]]], axis=0)
        .drop_duplicates()
        .sort_values("ds")
    )

    # 3. Predict
    forecast = m.predict(future)

    # 4. Align forecast with validation window
    val = val_df.copy()
    val["ds"] = pd.to_datetime(val["ds"])
    val = val.sort_values("ds")

    forecast_indexed = forecast.set_index("ds")
    y_pred = forecast_indexed.loc[val["ds"], "yhat"].values
    y_true = val["y"].values

    # 5. Metrics
    rmse = math.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)

    print(f"[Prophet] Validation window: {val['ds'].min().date()} → {val['ds'].max().date()}")
    print(f"[Prophet] Validation RMSE: {rmse:.2f}")
    print(f"[Prophet] Validation MAE : {mae:.2f}")

    # Return model + full forecast for plotting / saving
    return m, forecast, rmse, mae


In [11]:
prophet_model, prophet_forecast_full, prophet_rmse, prophet_mae = evaluate_prophet(
    train_df, val_df
)


14:41:11 - cmdstanpy - INFO - Chain [1] start processing
14:41:11 - cmdstanpy - INFO - Chain [1] done processing
14:41:11 - cmdstanpy - ERROR - Chain [1] error: code '3221225785' 


[Prophet] Fitting model with default optimizer...

[Prophet] ⚠ Optimization failed with default settings.
[Prophet] Error was:
 Error during optimization! Command 'E:\Research_Final\component3_service\prophet_env\Lib\site-packages\prophet\stan_model\prophet_model.bin random seed=91238 data file=C:\Users\ASUS\AppData\Local\Temp\tmpqhatlkvm\3jxy99hp.json init=C:\Users\ASUS\AppData\Local\Temp\tmpqhatlkvm\awdaweto.json output file=C:\Users\ASUS\AppData\Local\Temp\tmpqhatlkvm\prophet_modelojvh5c_e\prophet_model-20251123144111.csv method=optimize algorithm=newton iter=10000' failed: 
[Prophet] Retrying with relaxed convergence (require_converged=False)...


14:41:11 - cmdstanpy - INFO - Chain [1] start processing
14:41:11 - cmdstanpy - INFO - Chain [1] done processing
14:41:11 - cmdstanpy - ERROR - Chain [1] error: code '3221225785' 
Optimization terminated abnormally. Falling back to Newton.
14:41:11 - cmdstanpy - INFO - Chain [1] start processing
14:41:11 - cmdstanpy - INFO - Chain [1] done processing
14:41:11 - cmdstanpy - ERROR - Chain [1] error: code '3221225785' 


RuntimeError: Error during optimization! Command 'E:\Research_Final\component3_service\prophet_env\Lib\site-packages\prophet\stan_model\prophet_model.bin random seed=3085 data file=C:\Users\ASUS\AppData\Local\Temp\tmpqhatlkvm\bkd9tlsx.json init=C:\Users\ASUS\AppData\Local\Temp\tmpqhatlkvm\25harv66.json output file=C:\Users\ASUS\AppData\Local\Temp\tmpqhatlkvm\prophet_modelytck180w\prophet_model-20251123144111.csv method=optimize algorithm=newton iter=2000' failed: 

In [12]:
from prophet import Prophet
import pandas as pd
import numpy as np
import math
from sklearn.metrics import mean_squared_error, mean_absolute_error

def fit_prophet_model(train_df: pd.DataFrame):
    """
    Fit Prophet on train_df (expects columns: ds, y).
    - Cleans ds/y
    - Scales y to avoid numerical issues
    - Uses MCMC sampling (mcmc_samples > 0) instead of optimize
    Returns (model, scale_factor) or (None, None) on failure.
    """
    df = train_df.copy()[["ds", "y"]]

    # --- Clean ---
    df["ds"] = pd.to_datetime(df["ds"])
    df = df.sort_values("ds")

    # Ensure numeric y
    df["y"] = pd.to_numeric(df["y"], errors="coerce")
    df = df.dropna(subset=["ds", "y"])

    # If duplicates on ds, average them
    df = df.groupby("ds", as_index=False)["y"].mean()

    if len(df) < 10:
        print("[Prophet] Not enough points to fit (need at least ~10).")
        return None, None

    # --- Scale y to safer range ---
    max_y = df["y"].max()
    if max_y == 0 or np.isnan(max_y):
        print("[Prophet] y is zero or invalid; cannot fit model.")
        return None, None

    scale = max_y
    df_scaled = df.copy()
    df_scaled["y"] = df_scaled["y"] / scale

    # --- Build model: use sampling instead of optimizer ---
    m = Prophet(
        yearly_seasonality=True,
        weekly_seasonality=False,
        daily_seasonality=False,
        seasonality_mode="additive",
        mcmc_samples=300,   # <-- forces sampling path, avoids optimize()
    )

    try:
        print(f"[Prophet] Fitting model on {len(df_scaled)} points (scaled y)...")
        m.fit(df_scaled)
    except RuntimeError as e:
        print("[Prophet] ❌ Sampling failed:")
        print(e)
        return None, None

    return m, scale


def evaluate_prophet(train_df: pd.DataFrame, val_df: pd.DataFrame):
    """
    Fit Prophet on train_df, forecast over train+val window,
    and compute RMSE/MAE on validation.
    train_df, val_df must each have: ds (datetime), y (numeric).
    """
    # 1) Fit model (with scaling + MCMC)
    m, scale = fit_prophet_model(train_df)
    if m is None:
        print("[Prophet] Skipping evaluation because fit failed.")
        return None, None, None, None

    # 2) Clean validation data
    val = val_df.copy()[["ds", "y"]]
    val["ds"] = pd.to_datetime(val["ds"])
    val["y"] = pd.to_numeric(val["y"], errors="coerce")
    val = val.dropna(subset=["ds", "y"])
    val = val.sort_values("ds")
    val = val.groupby("ds", as_index=False)["y"].mean()

    if val.empty:
        print("[Prophet] Validation set is empty after cleaning.")
        return m, None, None, None

    # 3) Build future df = union of train + val ds
    train_clean = train_df.copy()[["ds"]]
    train_clean["ds"] = pd.to_datetime(train_clean["ds"])
    train_clean = train_clean.dropna().drop_duplicates()

    future = (
        pd.concat([train_clean[["ds"]], val[["ds"]]], axis=0)
        .drop_duplicates()
        .sort_values("ds")
    )

    # 4) Predict (yhat is in *scaled* units)
    forecast = m.predict(future)

    # 5) Rescale predictions back to original units
    forecast["yhat_rescaled"] = forecast["yhat"] * scale

    # Align with validation window
    forecast_idx = forecast.set_index("ds")
    y_pred = forecast_idx.loc[val["ds"], "yhat_rescaled"].values
    y_true = val["y"].values

    # 6) Metrics
    rmse = math.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)

    print(f"[Prophet] Validation window: {val['ds'].min().date()} → {val['ds'].max().date()}")
    print(f"[Prophet] Validation RMSE: {rmse:.2f}")
    print(f"[Prophet] Validation MAE : {mae:.2f}")

    return m, forecast, rmse, mae


In [13]:
prophet_model, prophet_forecast_full, prophet_rmse, prophet_mae = evaluate_prophet(
    train_df, val_df
)


14:42:58 - cmdstanpy - INFO - CmdStan start processing


[Prophet] Fitting model on 82 points (scaled y)...


chain 1:   0%|          | 0/300 [00:00<?, ?it/s, (Warmup)]


14:42:58 - cmdstanpy - ERROR - Chain [2] error: code '3221225785' 
14:42:58 - cmdstanpy - ERROR - Chain [1] error: code '3221225785' 
14:42:58 - cmdstanpy - ERROR - Chain [3] error: code '3221225785' 
14:42:58 - cmdstanpy - ERROR - Chain [4] error: code '3221225785' 
chain 2: 100%|██████████| 300/300 [00:00<00:00, 2742.46it/s, (Sampling completed)]

chain 3: 100%|██████████| 300/300 [00:00<00:00, 3008.89it/s, (Sampling completed)]


chain 4: 100%|██████████| 300/300 [00:00<00:00, 3035.55it/s, (Sampling completed)]


14:42:58 - cmdstanpy - INFO - CmdStan done processing.



[Prophet] ❌ Sampling failed:
Error during sampling:

Command and output files:
RunSet: chains=4, chain_ids=[1, 2, 3, 4], num_processes=4
 cmd (chain 1):
	['E:\\Research_Final\\component3_service\\prophet_env\\Lib\\site-packages\\prophet\\stan_model\\prophet_model.bin', 'id=1', 'random', 'seed=81733', 'data', 'file=C:\\Users\\ASUS\\AppData\\Local\\Temp\\tmpqhatlkvm\\lu5ekn24.json', 'init=C:\\Users\\ASUS\\AppData\\Local\\Temp\\tmpqhatlkvm\\2iyzcmir.json', 'output', 'file=C:\\Users\\ASUS\\AppData\\Local\\Temp\\tmpqhatlkvm\\prophet_modelyqfapaqg\\prophet_model-20251123144258_1.csv', 'method=sample', 'num_samples=150', 'num_warmup=150', 'algorithm=hmc', 'adapt', 'engaged=1']
 retcodes=[3221225785, 3221225785, 3221225785, 3221225785]
 per-chain output files (showing chain 1 only):
 csv_file:
	C:\Users\ASUS\AppData\Local\Temp\tmpqhatlkvm\prophet_modelyqfapaqg\prophet_model-20251123144258_1.csv
 console_msgs (if any):
	C:\Users\ASUS\AppData\Local\Temp\tmpqhatlkvm\prophet_modelyqfapaqg\prophet

In [14]:
from prophet import Prophet
import pandas as pd

df = pd.DataFrame({
    "ds": pd.date_range("2020-01-01", periods=24, freq="MS"),
    "y": range(24),
})

m = Prophet()
m.fit(df)
print("Fit OK")


14:44:26 - cmdstanpy - INFO - Chain [1] start processing
14:44:26 - cmdstanpy - INFO - Chain [1] done processing
14:44:26 - cmdstanpy - ERROR - Chain [1] error: code '3221225785' 


RuntimeError: Error during optimization! Command 'E:\Research_Final\component3_service\prophet_env\Lib\site-packages\prophet\stan_model\prophet_model.bin random seed=81888 data file=C:\Users\ASUS\AppData\Local\Temp\tmpqhatlkvm\lx77dciy.json init=C:\Users\ASUS\AppData\Local\Temp\tmpqhatlkvm\o5b49qf_.json output file=C:\Users\ASUS\AppData\Local\Temp\tmpqhatlkvm\prophet_modelv3xdl3rp\prophet_model-20251123144426.csv method=optimize algorithm=newton iter=10000' failed: 